In [3]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.tabddpm.models import Tabddpm
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "datasets").exists() and (p / "models").exists():
        ROOT = p
        break

raw_file = ROOT / "datasets" / "adult.csv"
processed_file = ROOT / "preprocessed_data" / "adult_tabddpm.csv"
artifact_dir = ROOT / "artifacts"

print("ROOT:", ROOT)
print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file),
    target_col="class"
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=Tabddpm()
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="adult_tabddpm",
    artifact_store=store,
    model_name="tabddpm",
)

print(results)

ROOT: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic
Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\adult.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\adult.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\adult_tabddpm.csv
Loaded data with shape: (32561, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved dataset artifact under datasets/adult_tabddpm/split-20260808-122629
Step 100/200 | MLoss: 26413.7109 | GLoss: 107971800.0000
Step 200/200 | MLoss: 24080.2383 | GLoss: 196731.2656
{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(da

In [4]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "adult_tabddpm"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = "class"

categorical_cols = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]

continuous_cols = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
]

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Synthetic type:", type(synthetic_df))
print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\adult_tabddpm\split-20260808-122629
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Synthetic sample:
        age  fnlwgt  education-num  capital-gain  capital-loss  \
0 -0.007528     3.0      -0.041774      0.909873      0.407949   
1 -0.182872     3.0      -0.070077      0.440771      0.237032   
2 -0.011564     3.0      -0.029124      0.985439      0.098734   
3 -0.111545     3.0      -0.033558      1.030666      0.776297   
4  0.110260     3.0       0.111852      1.349849      0.300539   

   hours-per-week workclass education      marital-status    occupation  \
0       -0.610834         ?      12th   Married-AF-spouse  Armed-Forces   
1       -0.421560         ?      12th   Married-AF-spouse  Adm-clerical   
2       -0.367324         ?      11th   Married-AF-spouse  Adm-clerical   
3       -0.611419         ?      12th  Married-civ-spouse  Armed-Forces   
4     